# 14. In-Field Protein Prediction Validation

Validate the best GPC prediction model (Random Forest, `mo_p1` temporal window) on
**186 independent in-field protein samples** from 16 wheat fields.

This tests whether the field-level model can predict **within-field** protein variability.

**Pipeline:**
1. Load GEE per-image S2 + daily ERA5 exports for in-field samples
2. Peak detection (GCVI double logistic) per field
3. Temporal alignment to peak-relative days
4. Aggregate to `mo_p1` window (days +16 to +45) with mean/std/p10 stats
5. Combine with elevation + soil features
6. Train RF on all field-level training data, predict on in-field samples
7. Evaluate R², RMSE, per-field analysis

In [1]:
import sys
import glob
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error

# Add project root to path
PROJECT_ROOT = Path('..').resolve()
DATA_ROOT = Path('/depot/ciampitti/data/WheatGrainProteinConcentrationPrediction')
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import load_config, get_feature_type
from src.data_loading import (
    load_spectral, load_meteo, load_elevation,
    load_static_features, merge_all_static
)
from src.peak_detection import detect_peaks
from src.temporal_alignment import (
    normalize_to_peak, aggregate_periods, compute_derived_meteo,
    generate_monthly_windows, _compute_agg
)

warnings.filterwarnings('ignore')

print('Project root:', PROJECT_ROOT)


Project root: /home/vmangidi/repositories/WheatGPCPipeline


## Configuration

In [ ]:
MODEL_PATH = PROJECT_ROOT / 'models/temporal_combinations/selected_monthly_analysis/models/mo_p1__to__mo_p1_RandomForest.joblib'

INFIELD_S2_DIR = DATA_ROOT / 'data/raw/spectral/GEE_Infield_Samples'
INFIELD_METEO_DIR = DATA_ROOT / 'data/raw/meteo/GEE_Infield_Samples'
INFIELD_ELEV_PATH = DATA_ROOT / 'data/raw/field_elevation_stats/infield_elevation_stats.csv'

INFIELD_METADATA_PATH = DATA_ROOT / 'data/processed/infield_samples_for_modeling.csv'
INFIELD_SOIL_PATH = DATA_ROOT / 'data/processed/two_layers_infield_samples_soil_features_saxton_by_field.csv'

config = load_config()
config['aggregation']['mode'] = 'monthly'

print('In-field S2 dir:', INFIELD_S2_DIR)
print('In-field meteo dir:', INFIELD_METEO_DIR)
print('Model path:', MODEL_PATH)


## 1. Load Model Metadata

In [3]:
model_data = joblib.load(MODEL_PATH)

selected_features = model_data['features']
best_params = model_data['best_params']

print(f"Model: {model_data['combination']} - RandomForest")
print(f"CV R2: {model_data['cv_r2']:.4f}")
print(f"CV RMSE: {model_data['cv_rmse']:.4f}")
print(f"\nSelected features ({len(selected_features)}):")
for i, f in enumerate(selected_features, 1):
    print(f"  {i:2d}. {f}")
print(f"\nBest params: {best_params}")

Model: mo_p1__to__mo_p1 - RandomForest
CV R2: 0.3037
CV RMSE: 1.1122

Selected features (18):
   1. elev_min
   2. mo_p1_B6_mean
   3. mo_p1_CVI_std
   4. mo_p1_GVI_p10
   5. mo_p1_LSWI_std
   6. mo_p1_MIRBI_mean
   7. mo_p1_MIRBI_p10
   8. mo_p1_MTCI_std
   9. mo_p1_NBR2_std
  10. mo_p1_NDSWIR1RedEdge1_p10
  11. mo_p1_S2REP_std
  12. mo_p1_dewpoint_mean
  13. mo_p1_t2m_max_mean
  14. soil_clay_ratio
  15. soil_sub_ksat_log10
  16. soil_subsoil_clay
  17. soil_topsoil_cec
  18. soil_topsoil_om

Best params: {}


## 2. Build Training Feature Matrix

Reconstruct the full training dataset using the pipeline modules.

In [ ]:
print('Loading training spectral data...')
train_spectral = load_spectral(config['data']['spectral_dir'])

print('Loading training meteo data...')
train_meteo = load_meteo(config['data']['meteo_dir'])

print('Running peak detection...')
train_peak_df, train_params_df = detect_peaks(train_spectral, config)

print('Normalizing to peak...')
train_spectral_norm = normalize_to_peak(train_spectral, train_peak_df)
train_meteo_norm = normalize_to_peak(train_meteo, train_peak_df)
train_meteo_norm = compute_derived_meteo(train_meteo_norm)

print('Aggregating periods...')
train_aggregated = aggregate_periods(train_spectral_norm, train_meteo_norm, config)

print('Loading static features...')
static_df, protein_df = load_static_features(
    config['data']['static_features_path'],
    config['soil']['columns'],
    config['soil'].get('static_columns')
)
elevation_df = load_elevation(config['data']['field_elevation_path'])
merged_static = merge_all_static(static_df, protein_df, elevation_df)

train_df = train_aggregated.merge(merged_static, on='field_key')

missing = [f for f in selected_features if f not in train_df.columns]
if missing:
    print(f'WARNING: Missing training features: {missing}')
else:
    print(f'All {len(selected_features)} features present in training data')

print(f'\nTraining data: {len(train_df)} fields x {len(train_df.columns)} columns')
train_df[selected_features + ['protein_pct']].describe().round(3)

## 3. Load In-Field Sample Metadata

In [5]:
# Load sample metadata with protein values
infield_meta = pd.read_csv(INFIELD_METADATA_PATH)
infield_meta['field_key'] = infield_meta['field_key'].astype(str)
infield_meta['sample_key'] = infield_meta['sample_key'].astype(str)

print(f'In-field samples: {len(infield_meta)}')
print(f'Fields: {infield_meta["field_key"].nunique()}')
print(f'Protein range: {infield_meta["protein_pct"].min():.1f} - {infield_meta["protein_pct"].max():.1f}%')
print(f'\nSamples per field:')
print(infield_meta.groupby('field_key').size().to_string())

In-field samples: 186
Fields: 16
Protein range: 10.2 - 16.3%

Samples per field:
field_key
1      12
10     12
11     12
12_    12
14      9
16     12
17     12
18     11
2      12
3      12
4      11
5      11
6      12
7      12
8      12
9      12


## 4. Load In-Field GEE Exports

**Prerequisite**: Run `src/gee_scripts/infield_samples_extraction.js` in GEE Code Editor
and download the exported CSVs from Google Drive.

In [6]:
# Load in-field S2 data (per-image observations)
s2_files = sorted(glob.glob(str(INFIELD_S2_DIR / 'infield_daily_s2_*.csv')))
print(f'Found {len(s2_files)} S2 files')

if not s2_files:
    raise FileNotFoundError(
        f'No S2 files found in {INFIELD_S2_DIR}. '
        'Download GEE exports from Google Drive first.'
    )

infield_s2 = pd.concat([pd.read_csv(f) for f in s2_files], ignore_index=True)
infield_s2['date'] = pd.to_datetime(infield_s2['date'])
infield_s2['field_key'] = infield_s2['field_key'].astype(str)
infield_s2['sample_key'] = infield_s2['sample_key'].astype(str)
infield_s2.sort_values(['sample_key', 'date'], inplace=True)
infield_s2.reset_index(drop=True, inplace=True)

# Drop rows where all spectral values are NaN (fully cloudy)
spectral_cols = [c for c in infield_s2.columns if c not in ['sample_key', 'field_key', 'date']]
before = len(infield_s2)
infield_s2 = infield_s2.dropna(subset=spectral_cols, how='all')
print(f'Dropped {before - len(infield_s2)} fully-cloudy rows')

print(f'\nIn-field S2 data: {len(infield_s2)} observations')
print(f'  Samples: {infield_s2["sample_key"].nunique()}')
print(f'  Fields: {infield_s2["field_key"].nunique()}')
print(f'  Date range: {infield_s2["date"].min().date()} to {infield_s2["date"].max().date()}')
print(f'  Unique dates: {infield_s2["date"].nunique()}')

Found 12 S2 files
Dropped 92933 fully-cloudy rows

In-field S2 data: 12157 observations
  Samples: 186
  Fields: 16
  Date range: 2024-07-05 to 2025-06-28
  Unique dates: 103


In [7]:
# Load in-field meteo data (daily)
meteo_files = sorted(glob.glob(str(INFIELD_METEO_DIR / 'infield_daily_meteo_*.csv')))
print(f'Found {len(meteo_files)} meteo files')

infield_meteo = pd.concat([pd.read_csv(f) for f in meteo_files], ignore_index=True)
infield_meteo['date'] = pd.to_datetime(infield_meteo['date'])
infield_meteo['field_key'] = infield_meteo['field_key'].astype(str)
infield_meteo['sample_key'] = infield_meteo['sample_key'].astype(str)
infield_meteo.sort_values(['sample_key', 'date'], inplace=True)
infield_meteo.reset_index(drop=True, inplace=True)

print(f'\nIn-field meteo data: {len(infield_meteo)} rows')
print(f'  Date range: {infield_meteo["date"].min().date()} to {infield_meteo["date"].max().date()}')
print(f'  Columns: {[c for c in infield_meteo.columns if c not in ["sample_key", "field_key", "date"]]}')

Found 12 meteo files

In-field meteo data: 67890 rows
  Date range: 2024-07-01 to 2025-06-30
  Columns: ['t2m_mean', 't2m_min', 't2m_max', 'precip', 'pet', 'gdd', 'hot_days', 'dewpoint']


In [8]:
# Load in-field elevation stats
infield_elev = pd.read_csv(INFIELD_ELEV_PATH)
infield_elev['field_key'] = infield_elev['field_key'].astype(str)
infield_elev['sample_key'] = infield_elev['sample_key'].astype(str)

print(f'Elevation data: {len(infield_elev)} samples')
print(f'Columns: {list(infield_elev.columns)}')
infield_elev.head()

Elevation data: 186 samples
Columns: ['sample_key', 'field_key', 'elev_mean', 'elev_min', 'slope_mean', 'aspect_mean']


,sample_key,field_key,elev_mean,elev_min,slope_mean,aspect_mean
0,14_6,14,686.505478,686.505478,1.896478,173.405727
1,14_7,14,686.887635,686.887635,2.017019,163.224847
2,14_5,14,689.199606,689.199606,1.498708,138.802005
3,14_4,14,686.600400,686.600400,1.211220,121.846220
4,14_2,14,691.301445,691.301445,2.057250,143.032491


In [ ]:
# The soil CSV uses a numeric sample_id that doesn't match sample_key — join via coordinates instead.
infield_soil = pd.read_csv(INFIELD_SOIL_PATH)
infield_soil['field_key'] = infield_soil['field_key'].astype(str)

soil_features = [
    'soil_clay_ratio', 'soil_sub_ksat_log10', 'soil_subsoil_clay',
    'soil_topsoil_cec', 'soil_topsoil_om'
]
available_soil = [c for c in soil_features if c in infield_soil.columns]
missing_soil = [c for c in soil_features if c not in infield_soil.columns]
if missing_soil:
    print(f'WARNING: Missing soil features: {missing_soil}')

infield_soil['lon_r'] = infield_soil['lon'].round(5)
infield_soil['lat_r'] = infield_soil['lat'].round(5)
meta_for_join = infield_meta[['sample_key', 'field_key', 'lon', 'lat']].copy()
meta_for_join['lon_r'] = meta_for_join['lon'].round(5)
meta_for_join['lat_r'] = meta_for_join['lat'].round(5)

infield_soil_clean = meta_for_join[['sample_key', 'lon_r', 'lat_r']].merge(
    infield_soil[['lon_r', 'lat_r'] + available_soil],
    on=['lon_r', 'lat_r'], how='left'
).drop(columns=['lon_r', 'lat_r'])

n_matched = infield_soil_clean[available_soil[0]].notna().sum()
print(f'Soil data: {n_matched}/{len(infield_soil_clean)} samples matched')
infield_soil_clean.head()


## 5. Peak Detection

Detect GCVI peak per field using double logistic fit.
Aggregate sample-level S2 data to field level for peak detection.

In [ ]:
field_s2 = infield_s2.groupby(['field_key', 'date']).mean(numeric_only=True).reset_index()
print(f'Field-level S2: {len(field_s2)} rows, {field_s2["field_key"].nunique()} fields')

infield_peak_df, infield_params_df = detect_peaks(field_s2, config)

print(f'\nPeaks detected for {len(infield_peak_df)} fields:')
for _, row in infield_peak_df.iterrows():
    flag = ' (ANOMALOUS)' if row.get('peak_anomalous', False) else ''
    print(f"  Field {row['field_key']}: peak = {row['peak_date']}{flag}")

fields_sorted = sorted(field_s2['field_key'].unique())
ncols = 4
nrows = (len(fields_sorted) + ncols - 1) // ncols

fig = make_subplots(
    rows=nrows, cols=ncols,
    shared_xaxes=True, shared_yaxes=True,
    subplot_titles=[f'Field {fk}' for fk in fields_sorted],
    horizontal_spacing=0.04, vertical_spacing=0.08
)

for idx, fk in enumerate(fields_sorted):
    row = idx // ncols + 1
    col = idx % ncols + 1
    grp = field_s2[field_s2['field_key'] == fk]

    fig.add_trace(go.Scatter(
        x=grp['date'], y=grp['GCVI'],
        mode='markers', marker=dict(size=4, opacity=0.5, color='steelblue'),
        showlegend=False,
        hovertemplate='%{x|%Y-%m-%d}<br>GCVI=%{y:.3f}<extra>Field ' + str(fk) + '</extra>'
    ), row=row, col=col)

    peak_row = infield_peak_df[infield_peak_df['field_key'] == fk]
    if len(peak_row) > 0:
        fig.add_vline(
            x=str(peak_row.iloc[0]['peak_date'])[:10],
            line=dict(color='red', dash='dash', width=1.5),
            row=row, col=col
        )

fig.update_layout(
    title='GCVI Profiles with Detected Peaks',
    height=250 * nrows, width=1100,
    template='plotly_white'
)
fig.show()


## 6. Temporal Alignment & Aggregation to mo_p1

Normalize dates to peak-relative days, then aggregate **per sample** within
the `mo_p1` window (days +16 to +45 relative to peak).

In [ ]:
infield_s2_norm = normalize_to_peak(infield_s2, infield_peak_df)
infield_meteo_norm = normalize_to_peak(infield_meteo, infield_peak_df)

print(f'Normalized S2: {len(infield_s2_norm)} rows')
print(f'Normalized meteo: {len(infield_meteo_norm)} rows')
print(f'Normalized day range (S2): '
      f'[{infield_s2_norm["normalized_day"].min()}, '
      f'{infield_s2_norm["normalized_day"].max()}]')

In [12]:
# Get mo_p1 window boundaries
mo_windows = generate_monthly_windows(config)
mo_p1_window = [w for w in mo_windows if w[0] == 'mo_p1'][0]
p1_name, p1_start, p1_end = mo_p1_window
print(f'mo_p1 window: days [{p1_start}, {p1_end}]')

# Filter to mo_p1 window
s2_p1 = infield_s2_norm[
    (infield_s2_norm['normalized_day'] >= p1_start) &
    (infield_s2_norm['normalized_day'] <= p1_end)
].copy()

meteo_p1 = infield_meteo_norm[
    (infield_meteo_norm['normalized_day'] >= p1_start) &
    (infield_meteo_norm['normalized_day'] <= p1_end)
].copy()

print(f'\nS2 observations in mo_p1: {len(s2_p1)}')
print(f'  Per sample: min={s2_p1.groupby("sample_key").size().min()}, '
      f'max={s2_p1.groupby("sample_key").size().max()}, '
      f'mean={s2_p1.groupby("sample_key").size().mean():.1f}')
print(f'\nMeteo observations in mo_p1: {len(meteo_p1)}')
print(f'  Per sample: min={meteo_p1.groupby("sample_key").size().min()}, '
      f'max={meteo_p1.groupby("sample_key").size().max()}, '
      f'mean={meteo_p1.groupby("sample_key").size().mean():.1f}')

mo_p1 window: days [16, 45]

S2 observations in mo_p1: 1243
  Per sample: min=3, max=16, mean=6.7

Meteo observations in mo_p1: 5580
  Per sample: min=30, max=30, mean=30.0


In [ ]:
agg_cfg = config['aggregation']['agg_functions']
exclude_cols = {'sample_key', 'field_key', 'date', 'normalized_day'}
s2_feature_cols = [c for c in s2_p1.columns if c not in exclude_cols]
meteo_feature_cols = [c for c in meteo_p1.columns if c not in exclude_cols]

all_sample_records = []

for sk in infield_meta['sample_key'].unique():
    record = {'sample_key': sk}

    s2_sample = s2_p1[s2_p1['sample_key'] == sk]
    for feat in s2_feature_cols:
        if feat not in s2_sample.columns:
            continue
        feat_type = get_feature_type(feat)
        stats = agg_cfg.get(feat_type, ['mean'])
        values = s2_sample[feat].dropna().values
        for stat in stats:
            record[f'mo_p1_{feat}_{stat}'] = _compute_agg(values, stat)

    meteo_sample = meteo_p1[meteo_p1['sample_key'] == sk]
    for feat in meteo_feature_cols:
        if feat not in meteo_sample.columns:
            continue
        feat_type = get_feature_type(feat)
        stats = agg_cfg.get(feat_type, ['mean'])
        values = meteo_sample[feat].dropna().values
        for stat in stats:
            record[f'mo_p1_{feat}_{stat}'] = _compute_agg(values, stat)

    all_sample_records.append(record)

infield_agg = pd.DataFrame(all_sample_records)
print(f'Aggregated features: {len(infield_agg)} samples x {len(infield_agg.columns) - 1} features')

spectral_meteo_features = [f for f in selected_features if f.startswith('mo_p1_')]
available = [f for f in spectral_meteo_features if f in infield_agg.columns]
missing = [f for f in spectral_meteo_features if f not in infield_agg.columns]
print(f'Model spectral/meteo features: {len(spectral_meteo_features)} | Available: {len(available)}')
if missing:
    print(f'Missing: {missing}')


## 7. Combine All Features

In [ ]:
infield_features = infield_agg.copy()

infield_features = infield_features.merge(
    infield_meta[['sample_key', 'field_key', 'protein_pct']],
    on='sample_key', how='left'
)

if 'elev_min' in infield_elev.columns:
    infield_features = infield_features.merge(
        infield_elev[['sample_key', 'elev_min']], on='sample_key', how='left'
    )
else:
    elev_temp = infield_elev[['sample_key']].copy()
    elev_temp['elev_min'] = infield_elev['elev_mean']
    infield_features = infield_features.merge(elev_temp, on='sample_key', how='left')
    print('WARNING: Using elev_mean as proxy for elev_min')

infield_features = infield_features.merge(infield_soil_clean, on='sample_key', how='left')

print('Feature availability check:')
for f in selected_features:
    if f in infield_features.columns:
        n_nan = infield_features[f].isna().sum()
        print(f'  {f:40s} OK   (NaN: {n_nan}/{len(infield_features)})')
    else:
        print(f'  {f:40s} MISSING')

print(f'Final in-field features: {len(infield_features)} samples')


## 8. Train Model & Predict

Train RandomForest on **all** field-level training data (no CV holdout),
then predict on in-field samples.

In [ ]:
train_valid = train_df.dropna(subset=selected_features + ['protein_pct'])
X_train = train_valid[selected_features].values
y_train = train_valid['protein_pct'].values
print(f'Training: {len(X_train)} fields')

infield_valid = infield_features.dropna(subset=selected_features + ['protein_pct'])
X_infield = infield_valid[selected_features].values
y_infield = infield_valid['protein_pct'].values
print(f'In-field: {len(X_infield)} samples '
      f'(dropped {len(infield_features) - len(infield_valid)} with NaN)')

rf = RandomForestRegressor(random_state=42, n_jobs=-1, **best_params)
rf.fit(X_train, y_train)
print(f'\nModel trained with {rf.n_estimators} trees')

y_pred = rf.predict(X_infield)

r2 = r2_score(y_infield, y_pred)
rmse = np.sqrt(mean_squared_error(y_infield, y_pred))
mbe = np.mean(y_pred - y_infield)

print(f'\n{"="*50}')
print(f'IN-FIELD VALIDATION RESULTS')
print(f'{"="*50}')
print(f'R2:   {r2:.4f}')
print(f'RMSE: {rmse:.4f} %')
print(f'MBE:  {mbe:.4f} %')
print(f'{"="*50}')

results_df = infield_valid[['sample_key', 'field_key', 'protein_pct']].copy()
results_df['predicted'] = y_pred
results_df['residual'] = y_pred - y_infield

## 9. Visualizations

In [16]:
plot_df = results_df.copy()
plot_df['field_key'] = plot_df['field_key'].astype(str)

fig = px.scatter(
    plot_df,
    x='protein_pct', y='predicted',
    color='field_key',
    hover_data={'sample_key': True, 'protein_pct': ':.2f', 'predicted': ':.2f', 'residual': ':.2f'},
    labels={'protein_pct': 'Observed Protein (%)', 'predicted': 'Predicted Protein (%)', 'field_key': 'Field'},
    title=f'In-Field Validation: RF mo_p1 — R²={r2:.3f}, RMSE={rmse:.3f}%, MBE={mbe:.3f}%',
    template='plotly_white',
    opacity=0.8
)

# 1:1 line
all_vals = list(plot_df['protein_pct']) + list(plot_df['predicted'])
lim_min, lim_max = min(all_vals) - 0.3, max(all_vals) + 0.3
fig.add_trace(go.Scatter(
    x=[lim_min, lim_max], y=[lim_min, lim_max],
    mode='lines', line=dict(color='black', dash='dash', width=1),
    name='1:1 line', showlegend=True
))

fig.update_layout(
    xaxis=dict(range=[lim_min, lim_max], scaleanchor='y', scaleratio=1),
    yaxis=dict(range=[lim_min, lim_max]),
    width=700, height=700,
    legend=dict(title='Field', font=dict(size=10))
)
fig.show()


In [ ]:
field_stats = results_df.groupby('field_key').agg(
    n_samples=('protein_pct', 'count'),
    obs_mean=('protein_pct', 'mean'),
    obs_std=('protein_pct', 'std'),
    pred_mean=('predicted', 'mean'),
    pred_std=('predicted', 'std'),
    rmse=('residual', lambda x: np.sqrt(np.mean(x**2))),
    mbe=('residual', 'mean'),
).round(3)

field_r2 = []
for fk, grp in results_df.groupby('field_key'):
    if len(grp) > 2 and grp['protein_pct'].std() > 0:
        field_r2.append({
            'field_key': fk,
            'r2': r2_score(grp['protein_pct'], grp['predicted'])
        })
    else:
        field_r2.append({'field_key': fk, 'r2': np.nan})

field_r2_df = pd.DataFrame(field_r2).set_index('field_key')
field_stats = field_stats.join(field_r2_df)

print('Per-field statistics:')
print(field_stats.to_string())
print(f'\nMean field RMSE: {field_stats["rmse"].mean():.3f}')
print(f'Mean field R2: {field_stats["r2"].mean():.3f}')

In [18]:
fig = make_subplots(rows=1, cols=2, subplot_titles=['Observed Protein (%) by Field', 'Predicted Protein (%) by Field'])

for fk in sorted(results_df['field_key'].unique()):
    fk_data = results_df[results_df['field_key'] == fk]
    fig.add_trace(go.Box(
        y=fk_data['protein_pct'], name=str(fk),
        marker_color='steelblue', showlegend=False,
        hovertemplate='Field ' + str(fk) + '<br>%{y:.2f}%<extra></extra>'
    ), row=1, col=1)
    fig.add_trace(go.Box(
        y=fk_data['predicted'], name=str(fk),
        marker_color='coral', showlegend=False,
        hovertemplate='Field ' + str(fk) + '<br>%{y:.2f}%<extra></extra>'
    ), row=1, col=2)

fig.update_layout(
    title='In-Field Protein Distribution by Field',
    xaxis_title='Field', yaxis_title='Protein (%)',
    xaxis2_title='Field', yaxis2_title='Protein (%)',
    width=1100, height=500,
    template='plotly_white'
)
fig.show()


In [19]:
fields_sorted = sorted(results_df['field_key'].unique())
n_fields = len(fields_sorted)
ncols = 4
nrows = (n_fields + ncols - 1) // ncols

fig = make_subplots(
    rows=nrows, cols=ncols,
    subplot_titles=[
        f'Field {fk} (n={len(results_df[results_df["field_key"]==fk])}, '
        f'R²={field_stats.loc[fk,"r2"]:.2f})'
        if fk in field_stats.index and np.isfinite(field_stats.loc[fk, 'r2'])
        else f'Field {fk} (R²=N/A)'
        for fk in fields_sorted
    ],
    horizontal_spacing=0.05, vertical_spacing=0.10
)

for idx, fk in enumerate(fields_sorted):
    row = idx // ncols + 1
    col = idx % ncols + 1
    fk_data = results_df[results_df['field_key'] == fk].sort_values('protein_pct').reset_index(drop=True)
    x = list(range(len(fk_data)))

    # Connecting lines
    for i, r in fk_data.iterrows():
        fig.add_trace(go.Scatter(
            x=[i, i], y=[r['protein_pct'], r['predicted']],
            mode='lines', line=dict(color='lightgray', width=1),
            showlegend=False, hoverinfo='skip'
        ), row=row, col=col)

    # Observed
    fig.add_trace(go.Scatter(
        x=x, y=fk_data['protein_pct'],
        mode='markers', marker=dict(color='steelblue', size=6, symbol='circle'),
        name='Observed', showlegend=(idx == 0),
        hovertemplate='Obs: %{y:.2f}%<extra></extra>'
    ), row=row, col=col)

    # Predicted
    fig.add_trace(go.Scatter(
        x=x, y=fk_data['predicted'],
        mode='markers', marker=dict(color='red', size=6, symbol='triangle-up'),
        name='Predicted', showlegend=(idx == 0),
        hovertemplate='Pred: %{y:.2f}%<extra></extra>'
    ), row=row, col=col)

fig.update_layout(
    title='Within-Field: Observed vs Predicted Protein per Sample',
    height=280 * nrows, width=1100,
    template='plotly_white',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
)
fig.show()


## 10. Feature Importance

In [20]:
importances = pd.Series(rf.feature_importances_, index=selected_features).sort_values(ascending=True)

fig = go.Figure(go.Bar(
    x=importances.values,
    y=importances.index,
    orientation='h',
    marker_color='steelblue',
    hovertemplate='%{y}<br>Importance: %{x:.4f}<extra></extra>'
))
fig.update_layout(
    title='Random Forest Feature Importance (MDI)',
    xaxis_title='Feature Importance (MDI)',
    yaxis_title='',
    width=700, height=550,
    template='plotly_white'
)
fig.show()


In [ ]:
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=['Residual Distribution', 'Residual vs Observed', 'Residuals by Field']
)

fig.add_trace(go.Histogram(
    x=results_df['residual'], nbinsx=30,
    marker_color='steelblue', opacity=0.75,
    hovertemplate='Residual: %{x:.2f}<br>Count: %{y}<extra></extra>',
    showlegend=False
), row=1, col=1)
fig.add_vline(x=0, line=dict(color='red', dash='dash'), row=1, col=1)

fig.add_trace(go.Scatter(
    x=results_df['protein_pct'], y=results_df['residual'],
    mode='markers', marker=dict(size=5, opacity=0.5, color='steelblue'),
    hovertemplate='Obs: %{x:.2f}%<br>Residual: %{y:.2f}%<extra></extra>',
    showlegend=False
), row=1, col=2)
fig.add_hline(y=0, line=dict(color='red', dash='dash'), row=1, col=2)

for fk in sorted(results_df['field_key'].unique()):
    fk_data = results_df[results_df['field_key'] == fk]
    fig.add_trace(go.Box(
        y=fk_data['residual'], name=str(fk),
        marker_color='steelblue', showlegend=False,
        hovertemplate='Field ' + str(fk) + '<br>%{y:.2f}%<extra></extra>'
    ), row=1, col=3)
fig.add_hline(y=0, line=dict(color='red', dash='dash'), row=1, col=3)

fig.update_layout(
    title='Residual Analysis',
    xaxis_title='Residual (Predicted − Observed)', yaxis_title='Count',
    xaxis2_title='Observed Protein (%)', yaxis2_title='Residual',
    xaxis3_title='Field', yaxis3_title='Residual',
    width=1200, height=450,
    template='plotly_white'
)
fig.show()

In [22]:
field_level = results_df.groupby('field_key').agg(
    obs_field_mean=('protein_pct', 'mean'),
    pred_field_mean=('predicted', 'mean'),
    n_samples=('sample_key', 'count'),
).reset_index()

r2_field = r2_score(field_level['obs_field_mean'], field_level['pred_field_mean'])
rmse_field = np.sqrt(mean_squared_error(field_level['obs_field_mean'], field_level['pred_field_mean']))

all_vals = list(field_level['obs_field_mean']) + list(field_level['pred_field_mean'])
lim_min, lim_max = min(all_vals) - 0.3, max(all_vals) + 0.3

fig = go.Figure()

# Bubbles sized by n_samples
fig.add_trace(go.Scatter(
    x=field_level['obs_field_mean'],
    y=field_level['pred_field_mean'],
    mode='markers+text',
    marker=dict(
        size=field_level['n_samples'] * 4,
        color='steelblue', opacity=0.7,
        line=dict(color='black', width=1)
    ),
    text=['F' + str(fk) for fk in field_level['field_key']],
    textposition='top right', textfont=dict(size=10),
    hovertemplate=(
        'Field %{text}<br>'
        'Observed: %{x:.2f}%<br>'
        'Predicted: %{y:.2f}%<br>'
        'n=%{marker.size}<extra></extra>'
    ),
    customdata=field_level['n_samples'],
    showlegend=False
))

# 1:1 line
fig.add_trace(go.Scatter(
    x=[lim_min, lim_max], y=[lim_min, lim_max],
    mode='lines', line=dict(color='black', dash='dash', width=1),
    name='1:1 line'
))

fig.update_layout(
    title=f'Field-Level Prediction (mean of in-field samples)<br>R²={r2_field:.3f}, RMSE={rmse_field:.3f}%',
    xaxis=dict(title='Observed Field Mean Protein (%)', range=[lim_min, lim_max], scaleanchor='y', scaleratio=1),
    yaxis=dict(title='Predicted Field Mean Protein (%)', range=[lim_min, lim_max]),
    width=650, height=650,
    template='plotly_white'
)
fig.show()

print(f'Field-level R2: {r2_field:.4f}')
print(f'Field-level RMSE: {rmse_field:.4f}%')


Field-level R2: -0.3605
Field-level RMSE: 1.3089%


## Summary

### Key Results
- **Model**: Random Forest, `mo_p1` temporal window, 18 features
- **Training**: Field-level CV R2 = 0.30 (228 fields)
- **In-field validation**: See metrics above

### Interpretation
- **Sample-level R2**: Can the model predict protein at individual sample locations?
- **Field-level R2**: Does averaging sample predictions recover field-level accuracy?
- **Within-field R2**: Can the model capture the relative protein ranking within a field?
- Per-field RMSE shows prediction accuracy varies by field